# Outings from the usual workplace: generation (binary logit)

A model of MAST (Maryland Activity Scheduling and Travel model), the activity-based model of the MILES platform, built on the ABIT code base.

## What this model answers

Does a worker leave the workplace for an outing during a work episode? The model uses work duration, arrival time and the workplace area.

| | |
|---|---|
| Choice | Outing or not |
| Alternatives | At least one outing; no outing (utility 0) |
| Decision makers | Workers at their usual workplace |
| Data | Regional Travel Survey 2017/18 (Baltimore and Washington regions), estimation half |

## How to run

- Kernel: *Python (biogeme)* (`~/.venvs/biogeme`, Biogeme 3.3). Then *Run All*.
- `RUN_DATA_PREP`: rebuild the records and tables in a scratch copy and compare them with the shipped ones.
- `RUN_CHECK`: log-likelihood at the reference coefficients, no estimation.
- `RUN_ESTIMATION`: run the Biogeme estimation (off by default).
- `RUN_HOLDOUT`: log-likelihood of the hold-out half.

## Contents

This notebook goes from the survey records to the estimates of this model:

1. Raw data: survey files, estimation and hold-out halves, the records of this model
2. Data preparation: filters from the survey to the records (records kept and dropped), rebuild check against the shipped tables
3. Choice set and estimation data
4. Descriptive statistics and figures; 4.1 every explanatory variable of the model in the data
5. Model specification (the folder's `model/estimate_outing_generation.py`, unchanged)
6. Correlation of the explanatory variables (heatmaps, VIF)
7. Log-likelihood at the reference coefficients
8. Biogeme estimation
9. Results: article format, what the coefficients say, fit, comparison with the article, correlation of the estimates
10. Hold-out sample
11. Glossary

## Model at a glance

**1. Who decides.** Workers at their usual workplace. Data: Regional Travel Survey 2017/18 (Baltimore and Washington regions), estimation half; 8,613 work episodes.

**2. What is chosen (dependent variable).** Outing or not. Alternatives: At least one outing; no outing (utility 0).

**3. Explanatory variables** (signs of the article estimates; terms specific to an alternative or purpose are listed per series):

| Group | Variable | Sign |
|---|---|---|
| Constant | Constant | − |
| Work episode | Work duration, 0-4 h (per h) | + |
| Work episode | Work duration, 4-7 h (per h) | + |
| Work episode | Secondary work episode (arrival from 15:00) | − |
| Time of day | Arrival at work before 7:00 | − |
| Time of day | Arrival at work from 10:00 | + |
| Workplace and commute | Workplace zone with at least 10,000 jobs per km2 | + |
| Workplace and commute | log(1 + home-workplace distance, km) | − |

**4. How person and household traits enter.** Traits enter with one coefficient per alternative, against the reference alternative whose utility is 0.

**5. How to read a coefficient.** Positive makes the alternative more likely, negative less likely; exp(coefficient) is the factor by which the odds change for one unit more.

In [ ]:
# switches (see How to run) and set-up: imports, folders, helper modules
RUN_DATA_PREP  = True    # rebuild the records and estimation tables in a scratch copy and compare with the shipped tables (about 1 min)
RUN_CHECK      = True    # log-likelihood at the reference coefficients
RUN_ESTIMATION = False   # Biogeme estimation, written to output/ (False: results are read from output/ if there are any)
RUN_HOLDOUT    = True    # log-likelihood of the hold-out half at the reference coefficients

import os, sys, json, glob, inspect, importlib, time, re
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from biogeme.results_processing import EstimationResults

FOLDER = os.path.abspath("")                       # this model folder (VS Code starts the notebook here)
sys.path.insert(0, os.path.join(FOLDER, "..", "..", "tools")); import nbtools as nt; import nb_reader as rd; import figs as fg; import abit_figs as af
from IPython.display import Markdown
sys.path.insert(0, os.path.join(FOLDER, "model"))
RUN = json.load(open(os.path.join(FOLDER, "RUN.json")))
OUT = os.path.join(FOLDER, "output"); os.makedirs(OUT, exist_ok=True)
NAME = "outing_generation"; MODEL_FOLDER = "11_outing_generation"
pd.set_option("display.max_rows", 200, "display.max_columns", 60, "display.width", 200)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print("model folder:", os.path.basename(FOLDER))

## 1. Raw data

*Where the records come from. Look for: the files and the record counts.*

### 1.1 The 2017/18 Regional Travel Survey

All MAST models are estimated on the 2017/18 RTS of the Baltimore and Washington regions (weekday travel day). Three files
are shipped in `../00_shared_data/rts_2017_18/raw`:

- `RTS_raw_flat_data.csv.gz`: the survey as received, one row per trip with its household, person and vehicle fields;
- `rts_trips_clean.csv.gz`: the trips after the survey cleaning (one row per trip, departure / arrival in minutes);
- `rts_tours.csv.gz`: home-based tours, work-based subtours and open tours built from the clean trips.

The cleaning script of the last two files is not shipped; the cleaned files are the starting point of the preparation here.
Race and ethnicity fields are not read.

In [ ]:
# 1.1 The 2017/18 Regional Travel Survey
flat, trips, tours = nt.rts_raw()
split = pd.read_csv(os.path.join(nt.ABIT_SHARED, "rts_2017_18", "split", "rts_split.csv"))
display(nt.rts_counts(flat, trips, tours, split))
show = ["tripid", "household_id", "person_id", "tripno", "o_activity", "d_activity", "dep_min", "arr_min", "travel_mode",
        "distance", "hhsize", "numvehicle", "hh_income_broad", "age", "employment_status", "student_status", "wttrdfin"]
trips[[c for c in show if c in trips]].head()

### 1.2 Estimation and hold-out halves

The households are split in two halves (`make_rts_split.py`): strata of home county x household size (1 to 5+), a seeded
shuffle inside each stratum (seed 2026), then alternate assignment. Every model is estimated on the estimation half; the
hold-out (validation) half is the other half of the same survey and is not read by the estimation. With `RUN_DATA_PREP` the
split is rebuilt into a scratch folder and compared with the shipped `rts_split.csv`.

In [ ]:
# 1.2 Estimation and hold-out halves
REBUILD = os.path.join(OUT, "rebuild_check")            # scratch folder: provided files are never overwritten
if RUN_DATA_PREP:
    rc = nt.run_quiet(f"{sys.executable} make_rts_split.py", cwd=nt.ABIT_PREP, env={"ABIT_REBUILT": REBUILD})
    display(nt.compare_frames(pd.read_csv(os.path.join(REBUILD, "rts_2017_18", "split", "rts_split.csv")), split,
                              key=["household_id"], name="rts_split.csv"))
else:
    print("skipped (RUN_DATA_PREP = False)")
hh = flat.drop_duplicates("household_id").set_index("household_id")
hh["half"] = split.set_index("household_id")["half"]
t = pd.crosstab(hh["hhsize"].clip(upper=5).rename("household size (5 = 5+)"), hh["half"])
t.loc["all"] = t.sum()
t

### 1.3 Records of this model

`input/code/estimate_subtours.py` (function `data`) writes three files per half into `input/data/raw`: `wbs_<half>` (one
row per work episode at the usual workplace of a worker, with the outing flag `y` and the model variables), `wbs_w_<half>`
(the same episodes with their survey fields) and `wbs_out_<half>` (one row per work-based outing: stops, zones, modes).
Model 11 uses the work episodes (`wbs`).

In [ ]:
# 1.3 Records of this model
CODE = os.path.join(FOLDER, "input", "code"); RAW = os.path.join(FOLDER, "input", "data", "raw")
HALVES = ["estimation", "validation"]
WBS = {h: pd.read_parquet(os.path.join(RAW, f"wbs_{h}.parquet")) for h in HALVES}
WW = {h: pd.read_parquet(os.path.join(RAW, f"wbs_w_{h}.parquet")) for h in HALVES}
WO = {h: pd.read_parquet(os.path.join(RAW, f"wbs_out_{h}.parquet")) for h in HALVES}
display(pd.DataFrame({h: {"work episodes (wbs)": len(WBS[h]), "with an outing (y = 1)": int(WBS[h].y.sum()), "outings (wbs_out)": len(WO[h])} for h in HALVES}))
WO["estimation"][["person_id", "n_stops", "leave", "ret", "purposes", "zones", "wz", "hz", "mode", "tour_mode", "w"]].head()

## 2. Data preparation

*From the raw records to the estimation sample. Look for: the records kept and dropped at each filter. With RUN_DATA_PREP the rebuilt tables must equal the shipped ones.*

### 2.1 From survey episodes to work episodes and outings

`estimate_subtours.data` takes the typed episodes of the half, finds the work-based outings (`tasha_distributions.subtours`:
work at the usual workplace, one or more discretionary episodes, again work at the usual workplace; the work episode and its
continuations are merged) and keeps the work episodes below. The filters are mirrored with the script's helpers.

In [ ]:
# Data preparation
sys.path.insert(0, CODE)
import te_common as TE, tasha_distributions as TD
steps = {}
for h in HALVES:
    e = TE.episodes(h).sort_values(["person_id", "tripno"], kind="stable").reset_index(drop=True)
    e = TD.subtours(e)
    s = [("episodes of persons in the survey population", len(e))]
    w = e[e.type.isin(["PW", "SW"])]; s.append(("work episodes at the usual workplace", len(w)))
    w = w[~w.wbs_cont]; s.append(("not a return to the workplace after an outing", len(w)))
    w = w[w.seg.eq("worker_usual")]; s.append(("worker with a usual workplace", len(w)))
    span = np.where(w.wbs_n > 0, w.merged_dur, w.dur)
    w = w[pd.notna(span) & (span > 0)]; s.append(("duration known", len(w)))
    w = w[(w.wz > 0) & (w.hz > 0)]; s.append(("home and workplace zone known", len(w)))
    steps[h] = s; print(f"{h}: {int((w.wbs_n > 0).sum()):,} of the {len(w):,} work episodes have an outing")
nt.funnel2(steps, {"work episodes at the usual workplace": "other episode types",
                   "not a return to the workplace after an outing": "continuation merged into the work episode",
                   "worker with a usual workplace": "segment other than worker_usual",
                   "duration known": "last episode of the day (no departure)",
                   "home and workplace zone known": "home or workplace tract outside the zone system"})

### 2.2 From work episodes to the estimation table

`a01_outing.load` adds person attributes (age, licence) of the survey population; no record is dropped. The choice `y = 1` when the work episode has at least one outing.

### 2.3 Rebuild check

In a scratch copy of `input/code` without the raw files, `estimate_subtours.data` rebuilds them from the episodes, then
`build_estimation_tables.py` runs, and every rebuilt table is compared with the shipped one. The shipped
`estimate_subtours.py` reads the survey population from `00_shared_data/survey_pop`; the files are in
`00_shared_data/rts_2017_18/survey_pop`, so the rebuild sets that folder (`ES.A3 = te_common.A3`) before calling `data`.

In [ ]:
# 2.3 Rebuild check
if RUN_DATA_PREP:
    t0 = time.time()
    SCR = nt.scratch_tree(FOLDER, raw_files=[])
    SCODE = os.path.join(SCR, "input", "code")

    nt.run('python3 -c "import te_common as T, estimate_subtours as ES; ES.A3 = T.A3; ES.data(\'estimation\'); ES.data(\'validation\')"', cwd=SCODE)
    nt.run("python3 build_estimation_tables.py", cwd=SCODE)
    res = pd.concat([nt.compare_trees(os.path.join(SCR, "input", "data", "raw"), RAW),
                     nt.compare_trees(os.path.join(SCR, "input", "data", "estimation"), os.path.join(FOLDER, "input", "data", "estimation")).assign(half="estimation"),
                     nt.compare_trees(os.path.join(SCR, "input", "data", "holdout"), os.path.join(FOLDER, "input", "data", "holdout")).assign(half="hold-out")],
                    ignore_index=True)
    nt.drop_scratch(SCR)
    print(f"rebuild and comparison: {(time.time() - t0) / 60:.1f} min")
    display(res)
    nt.rebuild_verdict(res, glob.glob(os.path.join(FOLDER, "model", "estimate_*.py"))[0])
else:
    print("skipped (RUN_DATA_PREP = False): the shipped tables are used")

## 3. Estimation table

*The table the model reads. Look for: the columns used and what they mean (data dictionary).*

Binary logit: 1 = the work episode has at least one outing, 0 = none.

In [ ]:
# Estimation table
EST_T = pd.read_parquet(os.path.join(FOLDER, "input", "data", "estimation", "estimation_table.parquet"))
HOLD_T = pd.read_parquet(os.path.join(FOLDER, "input", "data", "holdout", "estimation_table.parquet"))
print(f"estimation table: {EST_T.shape[0]:,} rows x {EST_T.shape[1]} columns; hold-out {HOLD_T.shape[0]:,} rows")
display(EST_T.head())
pd.read_csv(os.path.join(FOLDER, "input", "data", "estimation", "data_dictionary.csv"))

## 4. Descriptive statistics

*What the sample looks like. Look for: plausible shares and maps, and the chosen alternatives against the available ones.*

### Observed choice (unweighted and with the survey trip weight `w`, which the estimation does not use)

In [ ]:
# Descriptive statistics
LAB = {0: "no outing", 1: "at least one outing"}
display(nt.shares_table(EST_T.y.astype(int).map(LAB), EST_T.w))

In [ ]:
# Span of the work episode by outcome, arrival time, commute distance and number of outings; work episodes of 7 
fig, ax = nt.panels(4, ncol=4, w=3.5, h=2.8)
for y_, lab in LAB.items():
    ax[0].hist(EST_T.dur_h[EST_T.y == y_].clip(upper=14), bins=np.arange(0, 14.25, 0.5), histtype="step", lw=1.6, color=nt.OBS if y_ else nt.LIGHT, label=lab)
ax[0].set_xlabel("work-episode span (h)"); ax[0].set_ylabel("work episodes"); ax[0].legend(fontsize=7, frameon=False)
ax[1].hist(EST_T.arr / 60, bins=np.arange(0, 24.5, 0.5), color=nt.OBS); ax[1].set_xlabel("arrival at work (hour)"); ax[1].set_ylabel("work episodes")
ax[2].hist(EST_T.km_comm.clip(upper=100), bins=50, color=nt.OBS); ax[2].set_xlabel("home - workplace car km"); ax[2].set_ylabel("work episodes")
c = EST_T.nsub.clip(upper=3).value_counts().sort_index(); nt.bar(c, "outings per work episode (3 = 3+)", "work episodes", ax=ax[3])
plt.tight_layout(); plt.show()
band = pd.cut(EST_T.dur_h, [0, 4, 7, 9, 24], labels=["<4 h", "4-7 h", "7-9 h", "9+ h"])
print("outing share (%) by span:", (100 * EST_T.groupby(band, observed=False).y.mean()).round(1).to_dict())

Span of the work episode by outcome, arrival time, commute distance and number of outings; work episodes of 7 h or more have the most outings.

In [ ]:
# Person and workplace variables (share of work episodes with the flag, and the outing share among them); workpl
items = [("age_u30", "age under 30"), ("age_55p", "age 55+"), ("female", "female"), ("lic", "driving licence"), ("car0", "no car"),
         ("carshort", "fewer cars than drivers"), ("kids", "child under 16"), ("inc_low", "income < $50k"), ("inc_high", "income $100k+"),
         ("cbd_w", "workplace in a CBD zone"), ("sw", "arrives after 15:00")]
t = pd.DataFrame({"share of work episodes (%)": [100 * EST_T[c].mean() for c, _ in items],
                  "outing share if 1 (%)": [100 * EST_T.y[EST_T[c] == 1].mean() for c, _ in items]}, index=[l for _, l in items]).round(1)
display(t)
ww = WW["estimation"].loc[EST_T.row_index.values]
fig, ax = plt.subplots(1, 2, figsize=(13, 5.6))
nt.zone_map(ww.wz.value_counts(), "work episodes (workplace zone)", ax=ax[0], log=True)
r = ww.assign(y=EST_T.y.values).groupby("wz").y.agg(["mean", "size"]); r = r[r["size"] >= 5]
nt.zone_map(100 * r["mean"], "outing share (%), zones with 5+ episodes", ax=ax[1])
ax[0].set_title("workplaces", fontsize=10); ax[1].set_title("outing share", fontsize=10)
plt.tight_layout(); plt.show()

Person and workplace variables (share of work episodes with the flag, and the outing share among them); workplace zones and the outing share by zone.

### Estimation half vs hold-out half

In [ ]:
# Estimation half vs hold-out half
t = nt.half_comparison(EST_T.y.astype(int).map(LAB), HOLD_T.y.astype(int).map(LAB), "choice")
print(f"records: estimation {len(EST_T):,}, hold-out {len(HOLD_T):,}; largest difference of a share: {t['difference (points)'].abs().max():.1f} points")
t

### 4.1 Variables in the data

Each explanatory variable of the final specification as it enters the utility, interactions included (read from the model's own utility functions: the part of the utility that each coefficient multiplies; coefficients that share a label, e.g. the copies of a term for several purposes, are shown in one panel). Here: each variable by the chosen alternative, estimation half, unweighted. Person and household variables (the same value in every utility they enter) are shown as the mean or share among the records that chose each alternative; attributes of the alternatives (values that differ between alternatives, e.g. cost) as the mean at the chosen alternative against the mean where the alternative is available.

In [ ]:
# every explanatory variable of the specification by chosen alternative
import varplots as vp
MV_ = af.module(MODEL_FOLDER); r_ = MV_.read("estimation"); D_ = r_[0] if isinstance(r_, tuple) else r_
e_ = MV_.formulas() if hasattr(MV_, "formulas") else MV_.loglike(); e_ = e_["loglike"] if isinstance(e_, dict) else e_
U_, AV_, CH_ = vp.find_choice(e_); ALT_NAMES = af.ALTS[MODEL_FOLDER]
VP = vp.by_choice(U_, AV_, CH_, D_, vp.abit_labels(MODEL_FOLDER), ALT_NAMES)
display(Markdown(vp.choice_counts_line(vp.choices(CH_, D_), ALT_NAMES, 'work episodes')))
vp.by_choice_figures(VP, list(ALT_NAMES.values()), unit='work episodes')

How to read it: dark grey bars = mean (or share with value 1) among the records that chose the alternative, grey range = 25th-75th percentile, dashed line = all records; for attributes of the alternatives, the light bar is the mean where the alternative is available.

### Data sources

Where each source comes from and where it goes (paths from `manual_estimation/`). *verify*: no download link is recorded in the project. The full download steps are in `INSTRUCTIONS.pdf`.

**2017/18 Regional Travel Survey (Baltimore and Washington regions), household travel survey, weekday travel day** — Baltimore Metropolitan Council (BMC) and the National Capital Region Transportation Planning Board (TPB) at the Metropolitan Washington Council of Governments (MWCOG)
- Where: https://www.baltometro.org/ and https://www.mwcog.org/ (verify: no download page is documented in the project; the data are released on request)
- How: Request the 2017/18 Regional Travel Survey public-use files from BMC (Baltimore region) or MWCOG/TPB (Washington region). The project used one flat file in which every trip row carries its household and person fields (household, person, trip and vehicle tables joined): RTS_raw_flat_data.csv, with the codebook RTS_codebook_data_dictionary.xlsx (sheets Household, Person, Trip, Vehicle). The two cleaned files rts_trips_clean.csv (163,290 trips, deduplicated) and rts_tours.csv (57,627 home-based tours) were made by 01_clean_rts.py of the tour-based MITO project; that script is not on this machine, so these two files are shipped and cannot be rebuilt here. Survey weights: wthhfin, wtperfin, wttrdfin. If you receive separate household / person / trip / vehicle tables instead, join them on household_id, person_id (persno) and tripid to the flat layout of the codebook.
- Put in: 00_shared_data/raw/rts_2017_18 (the project's copies are provided in 00_shared_data/rts_2017_18/raw as .csv.gz)
- Files: `RTS_raw_flat_data.csv`, `RTS_codebook_data_dictionary.xlsx`, `rts_trips_clean.csv`, `rts_tours.csv`

**Maryland Statewide Transportation Model (MSTM) 2016 skims, 1,588 SMZ/RMZ zones** — Maryland Department of Transportation (MDOT SHA), via the project's MAST Maryland set-up (ABIT/input/mito_los/skims)
- Where: https://www.roads.maryland.gov/ (verify: MSTM data are provided by MDOT on request; no download page is documented)
- How: Obtain the MSTM 2016 skims in OMX format (matrix mat1, lookups origins / destinations): traveltime_auto.omx (AM-peak car time, min), distance_auto.omx (car distance, m), bus_tt.omx (bus time, min; 1000 = no path, 5 intrazonal), rail_tt00.omx (rail time, min; 0 = no rail path). convert_mstm_skims.py turns them into the csv.gz files the models read.
- Put in: 00_shared_data/raw/mstm_skims_2016
- Files: `traveltime_auto.omx`, `distance_auto.omx`, `bus_tt.omx`, `rail_tt00.omx`

**MSTM zone system: SMZ/RMZ polygons, zone table, MAST zone attributes** — ABIT/input/maryland/zone_attributes.csv (MAST Maryland set-up)
- Where: https://www.roads.maryland.gov/ (verify: provided with the MSTM)
- How: Zone polygons SMZ_RMZ_02152011inMSTM_EPSG26918.shp (+ .dbf .shx .prj; field id / SMZRMZ, 1,588 zones), zoneSystem.csv (ZoneId, COUNTYFIPS, ...), zone_attributes.csv (id, centroid x/y EPSG:26985, countyfips, region types, bbsr, area_km2;
- Put in: 00_shared_data/raw/mstm_zone_system
- Files: `SMZ_RMZ_02152011inMSTM_EPSG26918.shp/.dbf/.shx/.prj`, `zoneSystem.csv`, `zone_attributes.csv`

**Census 2010 Gazetteer file of census tracts (internal points)** — U.S. Census Bureau
- Where: https://www.census.gov/geographies/reference-files/time-series/geo/gazetteer-files.html (verify: the project copy I695_pricing/data/census/Gaz_tracts_national.txt has no download log)
- How: 2010 Gazetteer, Census Tracts, national file Gaz_tracts_national.txt (GEOID, INTPTLAT, INTPTLONG). Used by survey_population.tract_zone: zone of each tract's internal point.
- Put in: 00_shared_data/raw/census_gazetteer_2010
- Files: `Gaz_tracts_national.txt`

## 5. Model specification

*The utility functions, read from the model script; nothing is re-specified here.*

The folder's script `model/estimate_outing_generation.py`, imported as it is. Its utility functions:

In [ ]:
# Model specification
sys.path.insert(0, os.path.join(FOLDER, "model")); sys.modules.pop("estimate_outing_generation", None)   # the model folder's script (input/code may hold a namesake)
import estimate_outing_generation as M
src = inspect.getsource(M)
print(src[:src.find("def read(")])

In [ ]:
# Model specification
r = M.read("estimation")
df = r[0] if isinstance(r, tuple) else r
print(f"estimation table as read by the script: {df.shape[0]:,} rows, {df.shape[1]} columns")

## 6. Correlation of the explanatory variables

*Correlation of the explanatory variables. Look for: |r| of 0.7 or more and VIF of 10 or more; these make single coefficients less precise.*

|r| ≥ 0.7 and VIF ≥ 10 are listed. Interaction terms are correlated with their main effects by construction.

In [ ]:
# Correlation of the explanatory variables
f = M.formulas() if hasattr(M, "formulas") else M.loglike()
expr = f["loglike"] if isinstance(f, dict) else f
used = [c for c in nt.model_variables(expr) if c in df and not c.startswith("av_") and c not in ("y", "wesml", "w", "weight")]
Xu = nt.explanatory(df[used])
print(f"{len(used)} data columns enter the utilities")
Cu = nt.corr_heatmap(Xu, "Variables of the utilities")
display(nt.high_pairs(Cu)); display(nt.vif(Xu))

## 7. Log-likelihood at the reference coefficients

*Log-likelihood at the article coefficients, without estimation. Look for: the two numbers should be equal (to rounding).*

No estimation: the script's `evaluate("estimation", "full")`, i.e. the log-likelihood of the estimation half at the full-precision coefficients
of `model/reference_estimates.csv`. Target: LL(beta) of that file, -3061.493572214086 (article -3061.5;
N = 8613, K = 8).

In [ ]:
# Log-likelihood at the reference coefficients
LL_BETA = -3061.493572214086
def evaluate(half):
    here = os.getcwd()
    try:
        return M.evaluate(half, 'full') if "at" in inspect.signature(M.evaluate).parameters else M.evaluate(half)
    finally:
        os.chdir(here)

if RUN_CHECK:
    t0 = time.time(); ll_ref = evaluate("estimation")
    print(f"{(time.time() - t0) / 60:.1f} min")
    if isinstance(ll_ref, (int, float)) and LL_BETA is not None:
        print(f"LL at the reference coefficients {ll_ref:,.4f}; LL(beta) of reference_estimates.csv {LL_BETA:,.4f}; difference {ll_ref - LL_BETA:+.4f}")
else:
    print("skipped (RUN_CHECK = False)")

## 8. Biogeme estimation

*Biogeme estimation, only with RUN_ESTIMATION. Look for: convergence and the run time.*

The script's `estimate()`: Biogeme with its defaults (`output/biogeme.toml`), robust standard errors. Report in
`output/outing_generation.html` and `.yaml`. Not run here (`RUN_ESTIMATION = False`); a result in `output/` is read only if it is
newer than `model/reference_estimates.csv`.

In [ ]:
# Biogeme estimation
if RUN_ESTIMATION:
    here = os.getcwd(); t0 = time.time()
    try:
        M.estimate(OUT)
    finally:
        os.chdir(here)
    print(f"estimation time {time.time()-t0:.0f} s")
ref_time = os.path.getmtime(os.path.join(FOLDER, "model", "reference_estimates.csv"))
y = [f for f in sorted(glob.glob(os.path.join(OUT, "*.yaml")), key=os.path.getmtime) if "eval" not in os.path.basename(f) and os.path.getmtime(f) > ref_time]
results = EstimationResults.from_yaml_file(y[-1]) if y else None
print(f"results read from {os.path.basename(y[-1])}" if y else "no current result in output/: the estimate tables of section 9 are skipped")

## 9. Results

*The estimates and what they mean. Look for: coefficients equal to the article within rounding, and the readings below.*

### Estimates in the article format
Coefficient (stars: *** p < 0.01, ** p < 0.05, * p < 0.10, as in the article), robust s.e., p.

In [ ]:
# Results
if results is not None:
    est_t = nt.estimates_table(results)
    display(nt.article_format(est_t))

### What the coefficients say

One plain line per term (article estimates of `model/reference_estimates.csv`) and the coefficient plot.

In [ ]:
# plain readings of the coefficients and the coefficient plot (95 % robust intervals, grouped by block)
rows = af.coef_rows(MODEL_FOLDER)
display(rd.abit_readings(rows).style.hide(axis="index").set_properties(**{"text-align": "left"}))
fg.coef_figure(rows); plt.show()

How to read it: a point right of zero makes the alternative more likely; an interval that crosses zero is not significant; colours are the alternatives or purposes a term belongs to.

### Goodness of fit

In [ ]:
# Goodness of fit
if results is not None:
    display(nt.fit_statistics(results))

### Comparison with the article (`model/reference_estimates.csv`)

In [ ]:
# Comparison with the article (model/reference_estimates.csv)
if results is not None:
    REF = os.path.join(FOLDER, "model", "reference_estimates.csv")
    cmp = nt.compare(est_t, REF)
    cmp.to_csv(os.path.join(OUT, "comparison_with_article_notebook.csv"), index=False)
    print(f"{int(cmp.matches_article.sum())} of {len(cmp)} parameters match the article (coefficient and robust s.e. within the 4th-decimal rounding); "
          f"largest |difference|: coefficient {cmp.diff_coef.abs().max():.5f}, s.e. {cmp.diff_se.abs().max():.5f}")
    display(cmp.round({"coefficient": 4, "robust_se": 4, "p": 4, "diff_coef": 5, "diff_se": 5}))
    stats = nt.article_statistics(REF)
    if len(stats):
        stats["Biogeme"] = stats["statistic"].map({"LL(beta)": results.final_loglikelihood, "K": results.number_of_free_parameters, "N": 8613})
        display(stats)
    nt.plot_against_article(cmp, RUN.get("model", NAME))

### Correlation of the estimates
Robust correlation matrix of the estimated parameters; pairs with |r| ≥ 0.7 listed.

In [ ]:
# Correlation of the estimates
if results is not None:
    display(nt.parameter_correlation(results, "Robust correlation of the estimates"))

## 10. Hold-out sample

*Fit on the hold-out records, which the estimation did not use. Look for: a log-likelihood per record close to that of the estimation sample, and the maps.*

Log-likelihood of the hold-out half at the reference coefficients (`evaluate("holdout", "full")`).

In [ ]:
# Hold-out sample
if RUN_HOLDOUT:
    ll_hold = evaluate("holdout")
else:
    print("skipped (RUN_HOLDOUT = False)")

### Hold-out shares

Observed and model shares of the hold-out half (model: mean of the probabilities at the reference coefficients, simulated with this script's own log-likelihood).

In [ ]:
# observed against model shares on the hold-out half, overall and for one segment
t1, (title, t2, xl), n_hold = af.holdout_tables(MODEL_FOLDER)
t2 = t2[(t2.observed > 0.01) | (t2.model > 0.01)]
A = af.ALTS[MODEL_FOLDER]
fg.obs_model_figure([("All" if len(A) > 2 else t1.index[0], t1, "", "Hold-out records (%)"), (title, t2, xl, f"{t1.index[0] if len(A) == 2 else A[1]} (%)")]); plt.show()
print(f"hold-out records: {n_hold:,}")

How to read it: dark grey observed, orange model; equal bars mean the model reproduces the hold-out shares overall and within the segment. The hold-out half was also used in the specification search.

## Glossary

- **MAST**: Maryland Activity Scheduling and Travel model: the activity-based model of the MILES platform, built on the ABIT code base (Activity-Based Incremental Transport model).
- **reference alternative**: The alternative whose utility is fixed at 0; the coefficients of the others are differences against it.
- **log-likelihood**: Sum over decision makers of ln(probability of the observed choice). Closer to 0 is better.
- **hold-out**: The 20 % (residential, cars) or half (survey models) of the records not used for estimation. It was also used in the specification search, so its fit is not a fully independent test.
- **robust s.e.**: Standard error from the sandwich formula, valid when the model is not exactly right.
- **VIF**: Variance inflation factor: how much the correlation with other variables inflates a coefficient's variance; 10 or more is high.